## Weekly Plan
### Week 2
- Learn `requests` and build scripts that call public APIs.
- Parse and validate JSON responses.
- Add error handling (`try/except`, status code checks, retries).

### 1) Free API call (no authentication) with `http.client`
Using JSONPlaceholder: `https://jsonplaceholder.typicode.com/posts`

In [6]:
import http.client
import json

conn = http.client.HTTPSConnection("jsonplaceholder.typicode.com")
conn.request("GET", "/posts")
res = conn.getresponse()
data = res.read()

print("Status:", res.status)
parsed = json.loads(data.decode("utf-8"))
print("Records returned:", len(parsed))
print("First record keys:", list(parsed[0].keys()))
print("First body:", parsed[0]["body"])

Status: 200
Records returned: 100
First record keys: ['userId', 'id', 'title', 'body']
First body: quia et suscipit
suscipit recusandae consequuntur expedita et cum
reprehenderit molestiae ut ut quas totam
nostrum rerum est autem sunt rem eveniet architecto


### 2) `requests` version with status checks
No API key is required for this endpoint.

In [7]:
import requests

BASE_URL = "https://jsonplaceholder.typicode.com"

try:
    response = requests.get(f"{BASE_URL}/posts", timeout=20)
    print("HTTP status:", response.status_code)
    response.raise_for_status()
    payload = response.json()
    print("Payload type:", type(payload).__name__)
    print("Sample size:", len(payload))
except requests.exceptions.RequestException as exc:
    print(f"Request failed: {exc}")
except ValueError as exc:
    print(f"JSON parse failed: {exc}")

HTTP status: 200
Payload type: list
Sample size: 100


### 3) Parse and validate JSON response
Validate that the payload is a list of post objects before using fields.

In [8]:
def validate_posts_payload(payload: object) -> tuple[bool, str]:
    if not isinstance(payload, list):
        return False, "Payload is not a list"

    if not payload:
        return False, "Payload list is empty"

    first = payload[0]
    if not isinstance(first, dict):
        return False, "First record is not a dictionary"

    required_keys = {"userId", "id", "title", "body"}
    missing = required_keys.difference(first.keys())
    if missing:
        return False, f"Missing keys in first record: {sorted(missing)}"

    return True, "Payload looks valid"

try:
    ok, message = validate_posts_payload(payload)
    print("Validation:", message)

    if ok:
        print("Example post title:", payload[0]["title"])
except NameError:
    print("Run the previous requests cell first to define 'payload'.")

Validation: Payload looks valid
Example post title: sunt aut facere repellat provident occaecati excepturi optio reprehenderit


### 4) Robust retry pattern (`try/except`, retries, status checks)
Example uses `comments?postId=1` from JSONPlaceholder.

In [12]:
import time

def fetch_with_retries(url: str, retries: int = 3, timeout: int = 20):
    last_error = None

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, timeout=timeout)

            if response.status_code >= 500:
                raise requests.exceptions.HTTPError(
                    f"Server error {response.status_code}"
                )

            response.raise_for_status()
            return response.json()

        except (requests.exceptions.RequestException, ValueError) as exc:
            last_error = exc
            print(f"Attempt {attempt}/{retries} failed: {exc}")
            if attempt < retries:
                time.sleep(attempt)

    raise RuntimeError(f"All retry attempts failed. Last error: {last_error}")

try:
    comments_payload = fetch_with_retries(
        url=f"{BASE_URL}/comments?postId=1",
        retries=3,
        timeout=20,
    )
    print("Fetched comments count:", len(comments_payload))
    if comments_payload:
        print("First commenter email:", comments_payload[0].get("email"))
except RuntimeError as exc:
    print(exc)

Fetched comments count: 5
First commenter email: Eliseo@gardner.biz
